# MAI-Image-2 in Microsoft Foundry

https://microsoft.ai/news/today-were-announcing-3-new-world-class-mai-models-available-in-foundry/

MAI-Image-2 has turbocharged image generation performance and speed on Copilot after debuting as a top 3 model family on the Arena.ai leaderboard. Users experience at least 2x faster generation times on Foundry and Copilot with similar quality, based on real-world production traffic data. Phased rollouts are also underway in Bing and PowerPoint.

MAI-Image-2 was created with photographers, designers, and visual storytellers that demand natural lighting, accurate skin tones and texture, and clear in-image text for diagrams, layouts, and graphics. Once again, speed and quality don’t come at higher costs – MAI-Image-2 is offered at competitive price-to-performance.

Customers are already embracing MAI-Image-2 for creative work. WPP, one of the world’s largest marketing and communications groups, is among the first enterprise partners building with MAI-Image-2 at scale.

> https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai?tabs=python

## Model Card
https://microsoft.ai/pdf/MAI-Image-2-Model-Card.pdf
    

In [ ]:
import base64
import io
import math
import matplotlib.pyplot as plt
import os
import requests
import sys

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from datetime import datetime, timezone
from dotenv import load_dotenv
from pathlib import Path
from PIL import Image

In [ ]:
sys.version

In [ ]:
print(f"Today is {datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

In [ ]:
# For Entra ID access via Azure CLI login
from azure.identity import AzureCliCredential
from azure.core.exceptions import ClientAuthenticationError

scope = "https://cognitiveservices.azure.com/.default"
credential = AzureCliCredential()

try:
    token = credential.get_token(scope).token
except ClientAuthenticationError as ex:
    raise RuntimeError(
        "Azure CLI credential failed. Run 'az login' in a terminal and retry."
    ) from ex

In [ ]:
load_dotenv(".env")

endpoint = os.getenv("AZURE_FOUNDRY_ENDPOINT")
deployment_name = "MAI-Image-2e"

In [ ]:
IMAGES_DIR = "images"

os.makedirs(IMAGES_DIR, exist_ok=True)

In [ ]:
width, height = 768, 768

response = requests.post(
    url=f"{endpoint}/mai/v1/images/generations",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
    },
    json={
        "model": deployment_name,
        "prompt": "A photorealistic image of the Eiffel Tower in Paris",
        "width": width,
        "height": height,
    },
)

response.raise_for_status()
result = response.json()

In [ ]:
print(result["data"][0]["revised_prompt"])

In [ ]:
print(result["model"])

In [ ]:
print(result["size"])

In [ ]:
print(datetime.fromtimestamp(result["created"], tz=timezone.utc))

In [ ]:
image_data = next(
    (output for output in result.get("data", []) if "b64_json" in output),
    None,
)

In [ ]:
if image_data:
    output_path = os.path.join(IMAGES_DIR, "image.png")
    image_bytes = base64.b64decode(image_data["b64_json"])
    Path(output_path).write_bytes(image_bytes)
    print(f"Image saved to {output_path}")
    display(Image.open(io.BytesIO(image_bytes)))
else:
    print(f"Unexpected response format: {result}")

## Multiple tests

In [ ]:
def generate_image(
    endpoint: str,
    token: str,
    deployment_name: str,
    prompt: str,
    output_path: str = "image.png",
    width: int = 1024,
    height: int = 1024,
) -> Path | None:
    """Generate an image from a text prompt using the MAI image generation API.

    Args:
        endpoint: Base URL of the API endpoint.
        token: Bearer token for authentication.
        deployment_name: Name of the model deployment to use.
        prompt: Text description of the image to generate.
        output_path: File path where the generated image will be saved.
        width: Width of the generated image in pixels.
        height: Height of the generated image in pixels.

    Returns:
        Path to the saved image, or None if the response format was unexpected.
    """
    response = requests.post(
        url=f"{endpoint}/mai/v1/images/generations",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token}",
        },
        json={
            "model": deployment_name,
            "prompt": prompt,
            "width": width,
            "height": height,
        },
    )
    response.raise_for_status()
    result = response.json()

    image_data = next(
        (output for output in result.get("data", []) if "b64_json" in output),
        None,
    )

    if not image_data:
        print(f"Unexpected response format: {result}")
        return None

    image_bytes = base64.b64decode(image_data["b64_json"])
    image_file = Path(output_path)
    image_file.write_bytes(image_bytes)
    
    print(f"Image saved to {image_file}")

    return image_file

In [ ]:
prompts = [
    "A vast desert landscape with towering red rock formations at golden hour",
    "A minimalist modernist villa perched on a cliff overlooking the Mediterranean sea",
    "A majestic snow leopard",
    "People in a Paris street",
    "Beautiful Na'vi, avatar, photorealistic, james cameron, insane details",
    "A surrealist melting clock landscape inspired by Salvador Dali in photorealistic style",
    "A photorealistic close-up portrait of a weathered 70-year-old fisherman with deep blue eyes, grey stubble, and sun-beaten skin, shot on a Hasselblad with shallow depth of field",
    "A photorealistic studio portrait of a young woman with freckles and curly auburn hair",
]

In [ ]:
for prompt in prompts:
    generate_image(
        endpoint=endpoint,
        token=token,
        deployment_name=deployment_name,
        prompt=prompt,
        output_path=os.path.join(IMAGES_DIR, f"{'_'.join(prompt[:30].split())}.png"),
    )

In [ ]:
!ls $IMAGES_DIR -ls

In [ ]:
image_files = [
    os.path.join(IMAGES_DIR, f)
    for f in sorted(os.listdir(IMAGES_DIR))
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))
]

num_images = len(image_files)
ncols = 3
nrows = math.ceil(num_images / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = axes.flatten()

for ax, img_path in zip(axes, image_files):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(os.path.basename(img_path), fontsize=12)

for ax in axes[num_images:]:
    ax.axis("off")

plt.tight_layout()
plt.show()